In [12]:
import pandas as pd
from google.colab import files

# Upload CSV files in Colab
uploaded = files.upload()

# Load CSV files
audit_df = pd.read_csv("seo_audit.csv")
business_df = pd.read_csv("business_info.csv")

# Merge datasets using business_id
df = pd.merge(audit_df, business_df, on="business_id")

def calculate_score(row):
    score = 0
    issues = []

    # 1. Technical & SEO Content
    if row["is_https"]:
        score += 5
    else:
        issues.append("no HTTPS")

    if row["title_present"]:
        score += 5
    else:
        issues.append("missing Title Tag")

    if row["meta_description_present"]:
        score += 5
    else:
        issues.append("missing Meta Description")

    if row["h1_count"] > 0:
        score += 5
    else:
        issues.append(f"missing H1")

    if row["has_schema_markup"]:
        score += 5
    else:
        issues.append("no schema markup")

    if row["script_count"] < 40:
        score += 5
    else:
        issues.append(f"too many scripts (count: {row['script_count']})")

    # 2. Links
    if row["internal_links_count"] > 20:
        score += 10
    else:
        issues.append(f"low internal links (count: {row['internal_links_count']})")

    if row["external_links_count"] > 5:
        score += 5
    else:
        issues.append(f"low external links (count: {row['external_links_count']})")

    # 3. Lead Conversion info from business_df
    if pd.notna(row["phone"]):
        score += 10
    else:
        issues.append("no phone")

    if pd.notna(row["email"]):
        score += 10
    else:
        issues.append("no email")

    # 4. Social Presence
    social_count = 0
    if pd.notna(row["facebook"]): social_count += 1
    if pd.notna(row["instagram"]): social_count += 1
    if pd.notna(row["linkedin"]): social_count += 1

    if social_count == 3:
        score += 20
    elif social_count == 2:
        score += 14
    elif social_count == 1:
        score += 7
    else:
        issues.append("no social presence")

    # Priority
    if score <= 40:
        priority = "High"
    elif score <= 65:
        priority = "Medium"
    else:
        priority = "Low"

    # Reasoning: Include all factors/issues found with counts for numericals
    reasoning = ", ".join(issues) if issues else "No issues found"
    return pd.Series([score, reasoning, priority])

# Apply scoring
df[["score", "reasoning", "priority"]] = df.apply(calculate_score, axis=1)
df["lead_score"] = 100 - df["score"]

# Filter output: Keep only business info and results
audit_technical_cols = [c for c in audit_df.columns if c not in ['business_id']]
output_cols = [c for c in df.columns if c not in audit_technical_cols and c != 'score']
output_df = df[output_cols]

# Save and Download
output_df.to_csv("scored_leads.csv", index=False)
print("Scoring complete with detailed counts. Download the file below.")
display(output_df.head())
files.download("scored_leads.csv")

Saving business_info.csv to business_info (8).csv
Saving seo_audit.csv to seo_audit (8).csv
Scoring complete with detailed counts. Download the file below.


,id_x,business_id,id_y,email,phone,linkedin,instagram,facebook,reasoning,priority,lead_score
0,04241ab4-1e97-46ae-bdd2-982d76d04ecd,919d76b0-c781-4bee-8cff-161518f26b42,87b1113a-cf43-4026-b409-ee01991d5e06,NaN,NaN,NaN,NaN,NaN,"missing Title Tag, missing Meta Description, m...",High,90
1,042e6c6f-e80b-414a-9027-eee4ef99e9d8,fdafd4de-8d4d-4189-9037-92a3411830c0,8dd28ade-35ed-45a1-9d9b-d0a76935eb6e,NaN,(770) 299-1995,https://www.linkedin.com/company/raftrx/,https://www.instagram.com/raftrexteriors/,https://www.facebook.com/profile.php?id=615610...,"too many scripts (count: 54), no email",Low,30
2,05ae29b9-491c-47ed-afab-1cca59b1cd49,4fdf11d0-ce76-426b-8b64-8305abe7d360,41aeddbf-4bee-48c8-8cd5-7bc5bebe2b73,NaN,(713) 650-3022,https://www.linkedin.com/company/central-houst...,https://instagram.com/downtownhouston,https://www.facebook.com/pages/downtown-housto...,"no schema markup, no email",Low,30
3,07734ab8-5eb5-46e9-867f-9c8b85b6b2d4,c5d61e6e-f2ca-4096-9085-bf7e6e50b2b2,09f73479-740c-4019-b39c-ed67a5f4317f,NaN,NaN,NaN,NaN,NaN,"missing Title Tag, missing Meta Description, m...",High,90
4,0bc9293f-e2f8-419e-a2f1-c0784f78092e,78fcf75b-84ed-434f-b8d0-61c086a37148,756bc0aa-2669-4026-bd72-112c34a2f9d1,NaN,NaN,NaN,NaN,NaN,"missing Title Tag, missing Meta Description, m...",High,90


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
audit_df.columns

Index(['id', 'business_id', 'title_present', 'meta_description_present',
       'h1_count', 'has_schema_markup', 'is_https', 'script_count',
       'internal_links_count', 'external_links_count', 'extra_metrics',
       'created_at'],
      dtype='object')